# ゼロから作る Deep Learning ❸ 輪読会
## 第 4 ステージ「ニューラルネットワークを作る」 ― ステップ 47 〜 51

本ノートブックでは、書籍『ゼロから作る Deep Learning ❸ ― フレームワーク編』の第 4 ステージ、最終ブロック (ステップ 47 〜 51) の内容を取り扱う。ここでは、最終的に手書き数字認識 (MNIST) の予測を実行できるようになることが目標である。  

### これまでの内容 (ステップ 42 〜 46)
前回までに、DeZero に PyTorch と同様の機能を実装し、同じような操作で扱えるフレームワークに仕上げた。  
- `Parameter`/`Layer`/`Model`/`MLP` (パラメータ管理の自動化)
- `SGD`/`MomentumSGD` (最適化手法の分離)
- スパイラル・データセットを分類

学習の「型」も整備した。

```python
y = model(x); loss = loss_func(y, t)
model.cleargrads(); loss.backward(); optimizer.update()
```

### 今回の目標 ― 実践的なデータ処理と MNIST の予測
ここでは、実際のタスクで使う必要な機能を揃え、手書き数字認識を扱う。  

| ステップ | テーマ | 内容 |
|---|---|---|
| 47 | ソフトマックスと交差エントロピー誤差 | 多値分類の損失を DeZero の関数として実装 |
| 48 | 多値分類 | スパイラルを softmax_cross_entropy で分類 |
| 49 | Dataset クラスと前処理 | データを扱う統一的な仕組み |
| 50 | DataLoader | ミニバッチを自動で取り出すイテレータ |
| 51 | MNIST の学習 | 本物の手書き数字データで学習 |

このブロックを終えると、次のように PyTorch と同じような操作感で使用可能になる。  

```python
train_loader = DataLoader(train_set, batch_size=100)
for epoch in range(max_epoch):
    for x, t in train_loader:        # ミニバッチが自動で出てくる
        y = model(x)
        loss = softmax_cross_entropy(y, t)
        model.cleargrads(); loss.backward(); optimizer.update()
```


## 準備：これまでの DeZero を読み込む

このノートブックは単独で動くように、ステップ 46 までの DeZero (テンソル・NN 構築・Optimizer 対応) を最初にまとめて読み込む。

含まれるもの：`Variable`/`Function`、四則演算・累乗、テンソル関数 (`reshape`/`sum`/`matmul`/`linear` など)、`Parameter`/`Layer`/`LinearLayer`/`Model`/`MLP`、`Optimizer`/`SGD`/`MomentumSGD`。

In [ ]:
import numpy as np
import weakref
import contextlib
import matplotlib.pyplot as plt

# ===== ステップ 46 までの DeZero =====
class Config:
    enable_backprop = True
    train = True

@contextlib.contextmanager
def using_config(name, value):
    old_value = getattr(Config, name)
    setattr(Config, name, value)
    try:
        yield
    finally:
        setattr(Config, name, old_value)

def no_grad():
    return using_config('enable_backprop', False)
def test_mode():
    return using_config('train', False)

def as_array(x):
    if np.isscalar(x):
        return np.array(x)
    return x
def as_variable(obj):
    if isinstance(obj, Variable):
        return obj
    return Variable(obj)


class Variable:
    __array_priority__ = 200
    def __init__(self, data, name=None):
        if data is not None and not isinstance(data, np.ndarray):
            raise TypeError('{} is not supported'.format(type(data)))
        self.data = data
        self.name = name
        self.grad = None
        self.creator = None
        self.generation = 0
    def set_creator(self, func):
        self.creator = func
        self.generation = func.generation + 1
    def cleargrad(self):
        self.grad = None
    @property
    def shape(self): return self.data.shape
    @property
    def ndim(self): return self.data.ndim
    @property
    def size(self): return self.data.size
    def __len__(self): return len(self.data)
    def reshape(self, *shape):
        if len(shape) == 1 and isinstance(shape[0], (tuple, list)):
            shape = shape[0]
        return reshape(self, shape)
    @property
    def T(self): return transpose(self)
    def sum(self, axis=None, keepdims=False): return sum(self, axis, keepdims)
    def __getitem__(self, slices): return get_item(self, slices)
    def __repr__(self):
        if self.data is None: return 'variable(None)'
        p = str(self.data).replace('\n', '\n' + ' ' * 9)
        return 'variable(' + p + ')'
    def backward(self, retain_grad=False, create_graph=False):
        if self.grad is None:
            self.grad = Variable(np.ones_like(self.data))
        funcs = []
        seen_set = set()
        def add_func(f):
            if f not in seen_set:
                funcs.append(f); seen_set.add(f)
                funcs.sort(key=lambda x: x.generation)
        add_func(self.creator)
        while funcs:
            f = funcs.pop()
            gys = [output().grad for output in f.outputs]
            with using_config('enable_backprop', create_graph):
                gxs = f.backward(*gys)
                if not isinstance(gxs, tuple): gxs = (gxs,)
                for x, gx in zip(f.inputs, gxs):
                    if x.grad is None: x.grad = gx
                    else: x.grad = x.grad + gx
                    if x.creator is not None: add_func(x.creator)
            if not retain_grad:
                for y in f.outputs:
                    y().grad = None

class Parameter(Variable):
    pass

class Function:
    def __call__(self, *inputs):
        inputs = [as_variable(x) for x in inputs]
        xs = [x.data for x in inputs]
        ys = self.forward(*xs)
        if not isinstance(ys, tuple): ys = (ys,)
        outputs = [Variable(as_array(y)) for y in ys]
        if Config.enable_backprop:
            self.generation = max([x.generation for x in inputs])
            for output in outputs: output.set_creator(self)
            self.inputs = inputs
            self.outputs = [weakref.ref(output) for output in outputs]
        return outputs if len(outputs) > 1 else outputs[0]
    def forward(self, xs): raise NotImplementedError()
    def backward(self, gys): raise NotImplementedError()

class Add(Function):
    def forward(self, x0, x1):
        self.x0_shape, self.x1_shape = x0.shape, x1.shape
        return x0 + x1
    def backward(self, gy):
        gx0, gx1 = gy, gy
        if self.x0_shape != self.x1_shape:
            gx0 = sum_to(gx0, self.x0_shape); gx1 = sum_to(gx1, self.x1_shape)
        return gx0, gx1
class Mul(Function):
    def forward(self, x0, x1):
        self.x0_shape, self.x1_shape = x0.shape, x1.shape
        return x0 * x1
    def backward(self, gy):
        x0, x1 = self.inputs
        gx0, gx1 = gy * x1, gy * x0
        if self.x0_shape != self.x1_shape:
            gx0 = sum_to(gx0, self.x0_shape); gx1 = sum_to(gx1, self.x1_shape)
        return gx0, gx1
class Neg(Function):
    def forward(self, x): return -x
    def backward(self, gy): return -gy
class Sub(Function):
    def forward(self, x0, x1):
        self.x0_shape, self.x1_shape = x0.shape, x1.shape
        return x0 - x1
    def backward(self, gy):
        gx0, gx1 = gy, -gy
        if self.x0_shape != self.x1_shape:
            gx0 = sum_to(gx0, self.x0_shape); gx1 = sum_to(gx1, self.x1_shape)
        return gx0, gx1
class Div(Function):
    def forward(self, x0, x1):
        self.x0_shape, self.x1_shape = x0.shape, x1.shape
        return x0 / x1
    def backward(self, gy):
        x0, x1 = self.inputs
        gx0 = gy / x1; gx1 = gy * (-x0 / x1 ** 2)
        if self.x0_shape != self.x1_shape:
            gx0 = sum_to(gx0, self.x0_shape); gx1 = sum_to(gx1, self.x1_shape)
        return gx0, gx1
class Pow(Function):
    def __init__(self, c): self.c = c
    def forward(self, x): return x ** self.c
    def backward(self, gy):
        x, = self.inputs
        return self.c * x ** (self.c - 1) * gy
def add(x0,x1): return Add()(x0, as_array(x1))
def mul(x0,x1): return Mul()(x0, as_array(x1))
def neg(x): return Neg()(x)
def sub(x0,x1): return Sub()(x0, as_array(x1))
def rsub(x0,x1): return Sub()(as_array(x1), x0)
def div(x0,x1): return Div()(x0, as_array(x1))
def rdiv(x0,x1): return Div()(as_array(x1), x0)
def pow(x,c): return Pow(c)(x)
Variable.__add__=add; Variable.__radd__=add; Variable.__mul__=mul; Variable.__rmul__=mul
Variable.__neg__=neg; Variable.__sub__=sub; Variable.__rsub__=rsub
Variable.__truediv__=div; Variable.__rtruediv__=rdiv; Variable.__pow__=pow

class Reshape(Function):
    def __init__(self, shape): self.shape = shape
    def forward(self, x):
        self.x_shape = x.shape
        return x.reshape(self.shape)
    def backward(self, gy): return reshape(gy, self.x_shape)
def reshape(x, shape):
    if x.shape == shape: return as_variable(x)
    return Reshape(shape)(x)
class Transpose(Function):
    def forward(self, x): return np.transpose(x)
    def backward(self, gy): return transpose(gy)
def transpose(x): return Transpose()(x)
class BroadcastTo(Function):
    def __init__(self, shape): self.shape = shape
    def forward(self, x):
        self.x_shape = x.shape
        return np.broadcast_to(x, self.shape)
    def backward(self, gy): return sum_to(gy, self.x_shape)
def broadcast_to(x, shape):
    if x.shape == shape: return as_variable(x)
    return BroadcastTo(shape)(x)
def numpy_sum_to(x, shape):
    ndim = len(shape); lead = x.ndim - ndim
    lead_axis = tuple(range(lead))
    axis = tuple([i + lead for i, sx in enumerate(shape) if sx == 1])
    y = x.sum(lead_axis + axis, keepdims=True)
    if lead > 0: y = y.squeeze(lead_axis)
    return y
class SumTo(Function):
    def __init__(self, shape): self.shape = shape
    def forward(self, x):
        self.x_shape = x.shape
        return numpy_sum_to(x, self.shape)
    def backward(self, gy): return broadcast_to(gy, self.x_shape)
def sum_to(x, shape):
    if x.shape == shape: return as_variable(x)
    return SumTo(shape)(x)
def reshape_sum_backward(gy, x_shape, axis, keepdims):
    ndim = len(x_shape); tupled_axis = axis
    if axis is None: tupled_axis = None
    elif not isinstance(axis, tuple): tupled_axis = (axis,)
    if not (ndim == 0 or tupled_axis is None or keepdims):
        actual_axis = [a if a >= 0 else a + ndim for a in tupled_axis]
        shape = list(gy.shape)
        for a in sorted(actual_axis): shape.insert(a, 1)
    else: shape = gy.shape
    return gy.reshape(shape)
class Sum(Function):
    def __init__(self, axis, keepdims): self.axis=axis; self.keepdims=keepdims
    def forward(self, x):
        self.x_shape = x.shape
        return x.sum(axis=self.axis, keepdims=self.keepdims)
    def backward(self, gy):
        gy = reshape_sum_backward(gy, self.x_shape, self.axis, self.keepdims)
        return broadcast_to(gy, self.x_shape)
def sum(x, axis=None, keepdims=False): return Sum(axis, keepdims)(x)
class MatMul(Function):
    def forward(self, x, W): return x.dot(W)
    def backward(self, gy):
        x, W = self.inputs
        return matmul(gy, W.T), matmul(x.T, gy)
def matmul(x, W): return MatMul()(x, W)
class Linear(Function):
    def forward(self, x, W, b):
        y = x.dot(W)
        if b is not None: y += b
        return y
    def backward(self, gy):
        x, W, b = self.inputs
        gb = None if b.data is None else sum_to(gy, b.shape)
        gx = matmul(gy, W.T); gW = matmul(x.T, gy)
        return gx, gW, gb
def linear(x, W, b=None): return Linear()(x, W, b)

class Exp(Function):
    def forward(self, x): return np.exp(x)
    def backward(self, gy):
        y = self.outputs[0]()
        return gy * y
def exp(x): return Exp()(x)
def sigmoid(x):
    return 1 / (1 + exp(-x))

# Layer / Model / MLP / Optimizer
class Layer:
    def __init__(self): self._params = set()
    def __setattr__(self, name, value):
        if isinstance(value, (Parameter, Layer)): self._params.add(name)
        super().__setattr__(name, value)
    def __call__(self, *inputs):
        outputs = self.forward(*inputs)
        if not isinstance(outputs, tuple): outputs = (outputs,)
        self.inputs = [weakref.ref(x) for x in inputs]
        self.outputs = [weakref.ref(y) for y in outputs]
        return outputs if len(outputs) > 1 else outputs[0]
    def forward(self, inputs): raise NotImplementedError()
    def params(self):
        for name in self._params:
            obj = self.__dict__[name]
            if isinstance(obj, Layer): yield from obj.params()
            else: yield obj
    def cleargrads(self):
        for param in self.params(): param.cleargrad()

class LinearLayer(Layer):
    def __init__(self, out_size, nobias=False, in_size=None):
        super().__init__()
        self.in_size = in_size; self.out_size = out_size
        self.W = Parameter(None, name='W')
        if self.in_size is not None: self._init_W()
        self.b = None if nobias else Parameter(np.zeros(out_size), name='b')
    def _init_W(self):
        I, O = self.in_size, self.out_size
        self.W.data = np.random.randn(I, O) * np.sqrt(1 / I)
    def forward(self, x):
        if self.W.data is None:
            self.in_size = x.shape[1]; self._init_W()
        return linear(x, self.W, self.b)

class Model(Layer):
    def plot(self, *inputs): pass
class MLP(Model):
    def __init__(self, fc_output_sizes, activation=sigmoid):
        super().__init__()
        self.activation = activation; self.layers = []
        for i, out_size in enumerate(fc_output_sizes):
            layer = LinearLayer(out_size)
            setattr(self, 'l'+str(i), layer)
            self.layers.append(layer)
    def forward(self, x):
        for l in self.layers[:-1]:
            x = self.activation(l(x))
        return self.layers[-1](x)

class Optimizer:
    def __init__(self): self.target=None
    def setup(self, target): self.target=target; return self
    def update(self):
        params = [p for p in self.target.params() if p.grad is not None]
        for param in params: self.update_one(param)
    def update_one(self, param): raise NotImplementedError()
class SGD(Optimizer):
    def __init__(self, lr=0.01): super().__init__(); self.lr=lr
    def update_one(self, param): param.data -= self.lr * param.grad.data
class MomentumSGD(Optimizer):
    def __init__(self, lr=0.01, momentum=0.9):
        super().__init__(); self.lr=lr; self.momentum=momentum; self.vs={}
    def update_one(self, param):
        k=id(param)
        if k not in self.vs: self.vs[k]=np.zeros_like(param.data)
        v=self.vs[k]; v*=self.momentum; v-=self.lr*param.grad.data
        param.data += v


---
# ステップ 47：ソフトマックス関数と交差エントロピー誤差

前回までの内容では、`softmax_cross_entropy` を「とりあえず動くもの」として用意した。今回は、それを DeZero の関数として、逆伝播まで含めて実装する。まず準備として、スライス操作の関数から作る。  

## 47.1 スライス操作のための関数 get_item

多値分類では「各データの、正解クラスに対応する値」を取り出す操作が必要になる。これは配列のスライス (`x[行, 列]`) で行うが、逆伝播できる形で実装する必要がある。それが `get_item` 関数である。  

逆伝播の考え方：スライスは「一部を取り出す」操作である。その逆伝播は「取り出した場所に微分を戻し、それ以外は 0」にする。同じ場所が複数回取り出された場合は、微分を足し合わせる (`np.add.at` を使用)。  

In [ ]:
class GetItem(Function):
    def __init__(self, slices):
        self.slices = slices        # どこを取り出すか
    def forward(self, x):
        return x[self.slices]       # スライスで取り出す
    def backward(self, gy):
        x, = self.inputs
        f = GetItemGrad(self.slices, x.shape)
        return f(gy)

class GetItemGrad(Function):
    def __init__(self, slices, in_shape):
        self.slices = slices
        self.in_shape = in_shape    # 元の形状
    def forward(self, gy):
        gx = np.zeros(self.in_shape, dtype=gy.dtype)  # 元の形状のゼロ配列
        np.add.at(gx, self.slices, gy)  # 取り出した場所に微分を戻す (重複は加算)
        return gx
    def backward(self, ggx):
        return get_item(ggx, self.slices)

def get_item(x, slices):
    return GetItem(slices)(x)

# 動作確認：2 行目を取り出す
x = Variable(np.array([[1, 2, 3], [4, 5, 6]]))
y = get_item(x, 1)     # x[1] = [4,5,6]
print("x[1] =", y.data)

y.backward()
print("x.grad (取り出した 2 行目だけ 1)：")
print(x.grad.data)


2 行目を取り出したので、逆伝播では 2 行目だけが 1、1 行目は 0 になっている。`Variable` に `__getitem__` を設定してあるので、`x[1]` と普通のスライス記法でも書ける。  

## 47.2 ソフトマックス関数

ソフトマックス関数は、ニューラルネットワークの出力 (スコア) を確率 (0〜1、合計 1) に変換する。多値分類の出力層で使う。  

$$ \text{softmax}(x)_k = \frac{e^{x_k}}{\sum_{j=1}^{n} e^{x_j}} $$

式の解釈：各スコアを $e^{x}$ (指数関数) で「必ず正」にしてから、全体の合計で割って「合計 1」に正規化する。指数関数を使うことで、スコアの大小が確率の大小に強調して反映される。  

> 数値安定性：$e^x$ は $x$ が大きいとオーバーフロー (数値が大きすぎて計算できない) する。そこで、各要素から最大値を引いてから指数をとる。$\text{softmax}(x) = \text{softmax}(x - \max(x))$ が成り立つ (分子分母に同じ $e^{-\max}$ が掛かって打ち消される) ので、結果は変わらず安全に計算できる。

In [ ]:
def softmax(x, axis=1):
    x = as_variable(x)
    # 数値安定化：最大値を引いてから exp (結果は変わらない)
    y = exp(x - x.data.max(axis=axis, keepdims=True))
    sum_y = sum(y, axis=axis, keepdims=True)
    return y / sum_y     # 合計で割って確率にする

# 動作確認
x = Variable(np.array([[0.3, 2.9, 4.0]]))
p = softmax(x)
print("スコア：", x.data)
print("確率  ：", p.data)
print("合計  ：", p.data.sum(), " (1.0 になる)")


最大スコア (4.0) のクラスが最も高い確率になっている

## 47.3 交差エントロピー誤差

交差エントロピー誤差は、多値分類の損失関数である。「正解クラスの予測確率が高いほど小さくなる」損失である。

$$ L = -\frac{1}{N}\sum_{n=1}^{N} \log y_{n, t_n} $$

- $y_{n, t_n}$ は「$n$ 番目のデータの、正解クラス $t_n$ に対する予測確率」。
- $\log$ をとって符号を反転。正解確率が 1 (100%) なら $-\log 1 = 0$ (損失 0)、正解確率が小さいほど $-\log$ は大きくなる (損失大)。
- 全データで平均 ($\frac{1}{N}\sum$)。

式の解釈：「正解のクラスにどれだけ高い確率を割り当てられたか」を測る。正解に自信を持って正しく答えれば損失は小さく、正解を低く見積もれば損失は大きくなる。

実装では、ソフトマックスで確率にして、正解クラスの確率の `log` を取り出し、符号を反転して平均する。正解クラスの取り出しに、先ほどの `get_item` (スライス) を使う。

In [ ]:
class Log(Function):
    def forward(self, x): return np.log(x)
    def backward(self, gy):
        x, = self.inputs
        return gy / x            # log の微分は 1/x
def log(x): return Log()(x)

class Clip(Function):
    # 値を [x_min, x_max] に収める (log(0)=無限大 を防ぐため)
    def __init__(self, x_min, x_max):
        self.x_min = x_min; self.x_max = x_max
    def forward(self, x):
        return np.clip(x, self.x_min, self.x_max)
    def backward(self, gy):
        x, = self.inputs
        mask = (x.data >= self.x_min) * (x.data <= self.x_max)
        return gy * mask
def clip(x, x_min, x_max): return Clip(x_min, x_max)(x)

def softmax_cross_entropy(x, t):
    x, t = as_variable(x), as_variable(t)
    N = x.shape[0]
    p = softmax(x)                       # スコア → 確率
    p = clip(p, 1e-15, 1.0)              # log(0) 防止 (下限を設ける)
    log_p = log(p)                       # 確率の log
    # 各データの「正解クラス」の log 確率だけを取り出す (get_item)
    tlog_p = log_p[np.arange(N), t.data]
    y = -1 * sum(tlog_p) / N            # 符号反転して平均
    return y

# 動作確認
x = Variable(np.array([[0.3, 2.9, 4.0],     # データ 0 (正解はクラス 2)
                       [1.0, 1.0, 1.0]]))   # データ 1 (正解はクラス 0)
t = Variable(np.array([2, 0]))              # 正解ラベル
loss = softmax_cross_entropy(x, t)
print("損失：", loss.data)

loss.backward()
print("x.grad の形状：", x.grad.shape, " (勾配が計算できている)")


損失が計算でき、逆伝播も動いた。

> なぜ softmax と交差エントロピーをセットで使うのか：この 2 つを組み合わせると、逆伝播の勾配が $y - t$ (予測確率 − 正解) という分かりやすい形になる (`y` は softmax の出力確率、`t` は正解の one-hot)。「予測が正解からどれだけズレているか」がそのまま勾配になるので、学習が素直に進む。これは多値分類の定番の組み合わせである。

> ステップ 47 のまとめ
> - `get_item`：逆伝播できるスライス (取り出した場所に微分を戻す)。
> - `softmax`：スコアを確率に変換 (数値安定化のため最大値を引く)。
> - `softmax_cross_entropy`：多値分類の損失。正解クラスの確率が高いほど小さい。
> - この組み合わせは勾配が $y-t$ となり、学習が進みやすい。

---
# ステップ 48：多値分類

実装した `softmax_cross_entropy` を使って、前回のデータセットを改めて分類する。前回は損失関数を仮実装していたが、今回は正式な機能として、しかも `MLP` や `Optimizer` と組み合わせて書く。

## 48.1 スパイラル・データセット

データの用意 (前回と同じ)。

In [ ]:
def load_spiral(seed=1984):
    np.random.seed(seed)
    num_data, num_class, input_dim = 100, 3, 2
    data_size = num_data * num_class
    x = np.zeros((data_size, input_dim), dtype=np.float32)
    t = np.zeros(data_size, dtype=int)
    for j in range(num_class):
        for i in range(num_data):
            rate = i / num_data
            radius = 1.0 * rate
            theta = j * 4.0 + 4.0 * rate + np.random.randn() * 0.2
            ix = num_data * j + i
            x[ix] = np.array([radius * np.sin(theta), radius * np.cos(theta)]).flatten()
            t[ix] = j
    return x, t

x_data, t_data = load_spiral()
plt.figure(figsize=(5, 5))
for c, m in zip(range(3), ['o', 'x', '^']):
    mask = t_data == c
    plt.scatter(x_data[mask, 0], x_data[mask, 1], s=15, marker=m, label=f'class {c}')
plt.legend(); plt.title('spiral dataset'); plt.grid(alpha=0.3)
plt.show()


## 48.2 学習用のコード

`MLP`、`softmax_cross_entropy`、`SGD` を組み合わせた学習ループである。前回の実験とほぼ同じだが、今回は損失関数が正式機能になっている。ここでは、ミニバッチ学習を行う。

In [ ]:
# ハイパーパラメータ
lr = 1.0
max_epoch = 300
batch_size = 30
hidden_size = 10

x_data, t_data = load_spiral()
data_size = len(x_data)
max_iter = data_size // batch_size

np.random.seed(0)
model = MLP((hidden_size, 3))          # 隠れ 10 → 出力 3 クラス
optimizer = SGD(lr).setup(model)

loss_history = []
for epoch in range(max_epoch):
    idx = np.random.permutation(data_size)   # シャッフル
    x_shuffle, t_shuffle = x_data[idx], t_data[idx]

    sum_loss = 0
    for it in range(max_iter):
        bx = Variable(x_shuffle[it*batch_size:(it+1)*batch_size])
        bt = Variable(t_shuffle[it*batch_size:(it+1)*batch_size])

        y = model(bx)                              # 予測
        loss = softmax_cross_entropy(y, bt)        # 損失 (正式版！)
        model.cleargrads()
        loss.backward()
        optimizer.update()
        sum_loss += float(loss.data) * len(bt)

    avg_loss = sum_loss / data_size
    loss_history.append(avg_loss)
    if (epoch + 1) % 50 == 0:
        print(f"epoch {epoch+1:3d}: loss = {avg_loss:.3f}")


In [ ]:
# 決定境界と損失を可視化
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

h = 0.01
x_min, x_max = x_data[:,0].min()-0.1, x_data[:,0].max()+0.1
y_min, y_max = x_data[:,1].min()-0.1, x_data[:,1].max()+0.1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
with no_grad():
    score = model(Variable(grid))
pred = score.data.argmax(axis=1).reshape(xx.shape)

axes[0].contourf(xx, yy, pred, alpha=0.3, cmap='viridis')
for c, m in zip(range(3), ['o', 'x', '^']):
    mask = t_data == c
    axes[0].scatter(x_data[mask,0], x_data[mask,1], s=15, marker=m, label=f'class {c}')
axes[0].set_title('decision boundary'); axes[0].legend()

axes[1].plot(loss_history)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('loss')
axes[1].set_title('loss curve'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

pred = model(Variable(x_data)).data.argmax(axis=1)
print(f"最終正解率：{(pred == t_data).mean():.3f}")


正式な `softmax_cross_entropy` でも、きれいに分類できた。

> ステップ 48 のまとめ
> - 正式な `softmax_cross_entropy` で多値分類を実装。
> - `MLP` + 損失関数 + `Optimizer` で、学習コードがきれいに書ける。
> - スパイラル (3 クラス) を高い正解率で分類できた。

---
# ステップ 49：Dataset クラスと前処理

これまでは、データを NumPy 配列としてそのまま扱ってきた。しかし実際の機械学習では、データは巨大だったり、前処理 (正規化など) が必要だったりする。それらを統一的に扱うのが `Dataset` クラスである。

## 49.1 Dataset クラス

`Dataset` は「データを 1 つずつ取り出せる」共通の入れ物。ポイントは 2 つの特殊メソッドである。

- `__getitem__(index)`：`dataset[index]` で $\rm{index}$ 番目のデータ (入力とラベルのペア) を返す。
- `__len__()`：`len(dataset)` でデータ数を返す。

この 2 つを持つオブジェクトは、Python で「シーケンス (並び)」として扱える。あわせて前処理を組み合わせる仕組み (`transform`) も入れる。データを取り出すときに、指定した変換関数 (正規化など) を自動で適用する。

In [ ]:
class Dataset:
    def __init__(self, train=True, transform=None, target_transform=None):
        self.train = train
        # transform：入力データへの前処理。target_transform：ラベルへの前処理
        self.transform = transform if transform else (lambda x: x)
        self.target_transform = target_transform if target_transform else (lambda x: x)
        self.data = None
        self.label = None
        self.prepare()      # サブクラスでデータを読み込む

    def __getitem__(self, index):
        assert np.isscalar(index)   # 1 つのインデックスだけ想定
        if self.label is None:
            return self.transform(self.data[index]), None
        # 入力とラベルのペアを、前処理を適用して返す
        return self.transform(self.data[index]), self.target_transform(self.label[index])

    def __len__(self):
        return len(self.data)

    def prepare(self):
        pass   # サブクラスで、self.data と self.label を用意する


## 49.5 データセットの前処理

`Dataset` を継承して、具体的なデータセットを作る。ここでは、輪読会の時間内で学習が終わるよう、scikit-learn に含まれる手書き数字データ (8×8 ピクセル、1797 枚、10 クラス) を使う。これは MNIST (28×28) の簡易版で、オフラインでもすぐに使える。

`prepare` の中でデータを読み込み、`transform` で正規化 (ピクセル値を 0〜1 にする前処理) を行う。正規化すると学習が安定する。

In [ ]:
from sklearn.datasets import load_digits

class DigitsDataset(Dataset):
    def prepare(self):
        # 8x8 手書き数字データ (1797 枚) を読み込む
        digits = load_digits()
        X = digits.data.astype(np.float32)   # 形状 (1797, 64) 各画像は 64 次元に平坦化済み
        y = digits.target.astype(int)        # ラベル 0〜9
        n_train = 1500                        # 前 1500 枚を訓練、残りをテスト
        if self.train:
            self.data, self.label = X[:n_train], y[:n_train]
        else:
            self.data, self.label = X[n_train:], y[n_train:]

# 前処理：ピクセル値 (0〜16) を 0〜1 に正規化する関数
def normalize(x):
    return x / 16.0

# データセットを作る (前処理つき)
train_set = DigitsDataset(train=True, transform=normalize)
test_set = DigitsDataset(train=False, transform=normalize)

print("訓練データ数：", len(train_set))
print("テストデータ数：", len(test_set))

# 1 件取り出してみる
x, t = train_set[0]
print("1 件目：入力の形状", x.shape, ", ラベル", t)
print("入力の値の範囲：", x.min(), "〜", x.max(), " (正規化済み)")


実際の手書き数字画像を見てみる。64 次元のデータを 8×8 に戻して表示する。

In [ ]:
# 最初の 10 枚を表示
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    x, t = train_set[i]
    ax.imshow(x.reshape(8, 8), cmap='gray')
    ax.set_title(f'label: {t}')
    ax.axis('off')
plt.suptitle('handwritten digits (8x8)')
plt.tight_layout()
plt.show()


> ステップ 49 のまとめ
> - `Dataset` は `__getitem__`/`__len__` を持つデータの共通の入れ物。
> - `transform` で前処理 (正規化など) を取り出し時に自動適用。
> - 継承して `DigitsDataset` を作成。手書き数字を正規化して扱う。

---
# ステップ 50：ミニバッチを取り出す DataLoader

`Dataset` からミニバッチ (小分けにしたデータの束) を取り出す作業を自動化するのが `DataLoader` である。これを使えば、今まで手書きしていた「シャッフルして、batch_size ずつ切り出す」などの処理を、きれいにまとめられる。

## 50.1 イテレータとは

`DataLoader` は Python のイテレータとして作る。イテレータとは「`for` 文で 1 つずつ取り出せるオブジェクト」のことである。2 つの特殊メソッドで作れる。

- `__iter__()`：自分自身 (イテレータ) を返す。
- `__next__()`：次の要素を返す。もう無ければ `StopIteration` を送出して終了を知らせる。

これを実装すると、`for x, t in loader:` という自然な書き方でミニバッチを取り出せる。

In [ ]:
class DataLoader:
    def __init__(self, dataset, batch_size, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle              # 毎エポックでシャッフルするか
        self.data_size = len(dataset)
        self.max_iter = int(np.ceil(self.data_size / batch_size))  # 1 エポックの反復数
        self.reset()

    def reset(self):
        self.iteration = 0
        if self.shuffle:
            self.index = np.random.permutation(self.data_size)  # 順番をシャッフル
        else:
            self.index = np.arange(self.data_size)

    def __iter__(self):
        return self          # 自分自身がイテレータ

    def __next__(self):
        # 1 エポック分終わったら、リセットして終了を知らせる
        if self.iteration >= self.max_iter:
            self.reset()
            raise StopIteration

        i, bs = self.iteration, self.batch_size
        batch_index = self.index[i*bs:(i+1)*bs]        # 今回のバッチのインデックス
        batch = [self.dataset[int(idx)] for idx in batch_index]  # データを取り出す
        # 入力とラベルをそれぞれ束ねて NumPy 配列に
        x = np.array([example[0] for example in batch])
        t = np.array([example[1] for example in batch])

        self.iteration += 1
        return x, t

    def next(self):
        return self.__next__()

print("DataLoader を定義した")


## 50.2 DataLoader を使う

`DataLoader` を使うと、ミニバッチの取り出しが `for x, t in loader:` だけで書ける。

In [ ]:
train_loader = DataLoader(train_set, batch_size=100, shuffle=True)

# 1 エポック分、ミニバッチを取り出してみる
for i, (x, t) in enumerate(train_loader):
    print(f"バッチ {i}：x の形状 {x.shape}, t の形状 {t.shape}")
    if i >= 2:   # 最初の 3 バッチだけ表示
        print("  ...")
        break


## 50.3 accuracy 関数

学習がうまくいっているかを見るには、損失だけでなく正解率 (accuracy) も知りたい。予測 (最も確率の高いクラス) と正解ラベルが一致した割合を計算する関数を作る。

> `accuracy` は評価用の指標で、微分の必要はない (学習には使わない)。だから逆伝播は考えず、単純に計算する。

In [ ]:
def accuracy(y, t):
    y, t = as_variable(y), as_variable(t)
    # 予測クラス = 最も値が大きいクラス (argmax)
    pred = y.data.argmax(axis=1).reshape(t.shape)
    result = (pred == t.data)          # 正解と一致したか (True/False)
    acc = result.mean()                # 一致した割合
    return Variable(as_array(acc))

# 動作確認
y = Variable(np.array([[0.1, 0.9],     # 予測：クラス 1
                       [0.8, 0.2],     # 予測：クラス 0
                       [0.3, 0.7]]))   # 予測：クラス 1
t = Variable(np.array([1, 0, 0]))      # 正解：1, 0, 0
print("正解率：", accuracy(y, t).data, " (3 個中 2 個正解 → 0.667)")


> ステップ 50 のまとめ
> - `DataLoader` はイテレータ (`__iter__`/`__next__`) で、`for x,t in loader:` を実現。
> - シャッフルとミニバッチ切り出しを自動化。
> - `accuracy` で正解率を計算 (評価用、微分不要)。

---
# ステップ 51：MNIST (手書き数字) の学習

いよいよ、これまで作ってきた全部品を使って手書き数字認識を行う。書籍では MNIST (28×28、7 万枚) を使うが、本輪読会では時間の都合上、その簡易版である 8×8 手書き数字 (1797 枚) を使う。  

## 51.1 活性化関数 ReLU

MNIST のような本格的なタスクでは、活性化関数に ReLU (Rectified Linear Unit) がよく使われる。シグモイドより学習が速く進むことが多いためである。  

$$ \text{ReLU}(x) = \max(0, x) = \begin{cases} x & (x > 0) \\ 0 & (x \le 0) \end{cases} $$

式の解釈：「入力が正ならそのまま通し、負なら 0 にする」だけのシンプルな関数である。逆伝播も簡単で、順伝播で正だった要素は微分 1 (そのまま流す)、負だった要素は微分 0 (流さない)。シグモイドと違って、正の領域で微分が小さくならないので、深いネットワークでも学習が進みやすいのが利点である。

In [ ]:
class ReLU(Function):
    def forward(self, x):
        return np.maximum(x, 0.0)     # 負を 0 に、正はそのまま
    def backward(self, gy):
        x, = self.inputs
        mask = x.data > 0             # 順伝播で正だった場所だけ True
        return gy * mask             # 正だった要素だけ微分を流す

def relu(x):
    return ReLU()(x)

# ReLU の形と逆伝播を確認
x = Variable(np.array([[-2.0, -1.0, 0.0, 1.0, 2.0]]))
y = relu(x)
print("入力  ：", x.data)
print("ReLU  ：", y.data, " (負は 0 に)")
y.backward()
print("勾配  ：", x.grad.data, " (正だった場所だけ 1)")


## 51.2 MNIST の学習

すべての機能 (`DigitsDataset`, `DataLoader`, `MLP`, `ReLU`, `softmax_cross_entropy`, `MomentumSGD`, `accuracy`) を組み合わせて学習する。

ネットワークは「64 次元 (8×8 画像) → 隠れ層 100 (ReLU) → 10 クラス」の 2 層 MLP である。この学習ループが、今回の集大成である。

In [ ]:
# 前ステップで実装した softmax_cross_entropy を再掲 (このセルで使うため)
class Log(Function):
    def forward(self, x): return np.log(x)
    def backward(self, gy):
        x, = self.inputs
        return gy / x
def log(x): return Log()(x)
class Clip(Function):
    def __init__(self, x_min, x_max): self.x_min=x_min; self.x_max=x_max
    def forward(self, x): return np.clip(x, self.x_min, self.x_max)
    def backward(self, gy):
        x, = self.inputs
        mask = (x.data >= self.x_min) * (x.data <= self.x_max)
        return gy * mask
def clip(x, x_min, x_max): return Clip(x_min, x_max)(x)
def softmax(x, axis=1):
    x = as_variable(x)
    y = exp(x - x.data.max(axis=axis, keepdims=True))
    return y / sum(y, axis=axis, keepdims=True)
def softmax_cross_entropy(x, t):
    x, t = as_variable(x), as_variable(t)
    N = x.shape[0]
    p = clip(softmax(x), 1e-15, 1.0)
    log_p = log(p)
    tlog_p = log_p[np.arange(N), t.data]
    return -1 * sum(tlog_p) / N

# ===== ハイパーパラメータ =====
max_epoch = 30
batch_size = 100
hidden_size = 100
lr = 0.3

# データローダー
train_set = DigitsDataset(train=True, transform=normalize)
test_set = DigitsDataset(train=False, transform=normalize)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

# モデル (64→100→10) と最適化手法
np.random.seed(0)
model = MLP((hidden_size, 10), activation=relu)   # 活性化は ReLU
optimizer = MomentumSGD(lr=lr).setup(model)

train_loss_list, test_acc_list = [], []
for epoch in range(max_epoch):
    # --- 訓練 ---
    sum_loss = 0
    for x, t in train_loader:
        y = model(Variable(x))
        loss = softmax_cross_entropy(y, Variable(t))
        model.cleargrads()
        loss.backward()
        optimizer.update()
        sum_loss += float(loss.data) * len(t)
    avg_loss = sum_loss / len(train_set)
    train_loss_list.append(avg_loss)

    # --- テストデータで評価 (学習には使わないので no_grad) ---
    sum_acc, n = 0, 0
    with no_grad():
        for x, t in test_loader:
            y = model(Variable(x))
            acc = accuracy(y, Variable(t))
            sum_acc += float(acc.data) * len(t); n += len(t)
    test_acc = sum_acc / n
    test_acc_list.append(test_acc)

    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch+1:2d}: train_loss = {avg_loss:.3f}, test_acc = {test_acc:.3f}")

print(f"\n最終テスト正解率：{test_acc_list[-1]:.3f}")


手書き数字を高い正解率で認識できた。学習曲線を可視化する。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(train_loss_list)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('train loss')
axes[0].set_title('training loss'); axes[0].grid(alpha=0.3)

axes[1].plot(test_acc_list, color='green')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('test accuracy')
axes[1].set_title('test accuracy'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 追加実験：予測結果と「間違えた数字」を見る

学習したモデルが、実際にどんな予測をするのか見てみる。正しく認識できた例と、間違えた例を並べて表示する。間違えた数字を見ると、「人間でも紛らわしい形」を間違えていることが多い。

In [ ]:
# テストデータ全体で予測
X_test = np.array([test_set[i][0] for i in range(len(test_set))])
t_test = np.array([test_set[i][1] for i in range(len(test_set))])
with no_grad():
    pred = model(Variable(X_test)).data.argmax(axis=1)

correct_idx = np.where(pred == t_test)[0]
wrong_idx = np.where(pred != t_test)[0]
print(f"正解：{len(correct_idx)} 件, 不正解：{len(wrong_idx)} 件")

# 正解例5つ
fig, axes = plt.subplots(2, 5, figsize=(11, 4.5))
for i, ax in enumerate(axes[0]):
    idx = correct_idx[i]
    ax.imshow(X_test[idx].reshape(8, 8), cmap='gray')
    ax.set_title(f'O pred:{pred[idx]}', color='green', fontsize=10)
    ax.axis('off')
# 間違えた例5つ
for i, ax in enumerate(axes[1]):
    if i < len(wrong_idx):
        idx = wrong_idx[i]
        ax.imshow(X_test[idx].reshape(8, 8), cmap='gray')
        ax.set_title(f'X pred:{pred[idx]} true:{t_test[idx]}', color='red', fontsize=9)
    ax.axis('off')
plt.suptitle('top: correct,   bottom: mistakes')
plt.tight_layout()
plt.show()


> ステップ 51 のまとめ
> - `ReLU` ($\max(0,x)$) を実装。深いネットワークで学習が進みやすい活性化関数。
> - 全部品を組み合わせ、手書き数字認識を実現 (高い正解率)。
> - `Dataset`/`DataLoader` による実践的な学習ループが完成。

---
# 発展：PyTorch ではどう書く？

今回作った `Dataset`/`DataLoader`/`ReLU`/`softmax_cross_entropy` は、PyTorch にも対応するものがある。実際、DeZero のこれらは PyTorch を参考に設計されている。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader as TorchLoader, TensorDataset

# DeZero の DigitsDataset/DataLoader に対応
from sklearn.datasets import load_digits
digits = load_digits()
X = torch.tensor(digits.data / 16.0, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)
dataset = TensorDataset(X[:1500], y[:1500])
loader = TorchLoader(dataset, batch_size=100, shuffle=True)  # DeZero: DataLoader(...)

# DeZero の MLP((100,10), relu) に対応
model = nn.Sequential(nn.Linear(64, 100), nn.ReLU(), nn.Linear(100, 10))
optimizer = torch.optim.SGD(model.parameters(), lr=0.3, momentum=0.9)  # MomentumSGD

# 1エポック学習
for x, t in loader:
    y = model(x)                          # DeZero: model(x)
    loss = F.cross_entropy(y, t)          # DeZero: softmax_cross_entropy(y, t)
    optimizer.zero_grad()                 # DeZero: model.cleargrads()
    loss.backward()                       # DeZero: loss.backward()
    optimizer.step()                      # DeZero: optimizer.update()
print("[PyTorch] 1 エポック学習できた。loss =", round(loss.item(), 3))


### DeZero と PyTorch の対応表 (第 4 ステージ最終ブロック)

| 概念 | DeZero (手作り) | PyTorch |
|---|---|---|
| データセット | `Dataset` | `torch.utils.data.Dataset` |
| ミニバッチ供給 | `DataLoader` | `torch.utils.data.DataLoader` |
| スライス (逆伝播対応) | `get_item` | テンソルのスライスが自動対応 |
| ソフトマックス | `softmax` | `F.softmax` |
| 多値分類の損失 | `softmax_cross_entropy` | `F.cross_entropy` |
| ReLU | `relu` | `F.relu` / `nn.ReLU` |
| 正解率 | `accuracy` | 自作 / `torchmetrics` |

PyTorch を使うとき当たり前に書く `for x, t in loader:` や `F.cross_entropy` が、内部で何をしているかを、DeZero を手作りしたことで完全に理解できた。特に、`DataLoader` がイテレータであること、`cross_entropy` が softmax と log の組み合わせであることなど、普段は隠れている仕組みが見えるようになった。

---
# まとめ ― 第 4 ステージ (ニューラルネットワークを作る)

ステップ 47 〜 51 で、DeZero は本物のデータセットを学習できる実用的なディープラーニングフレームワークになった。

1. ステップ 47 (softmax/交差エントロピー) `get_item` (逆伝播対応スライス)、`softmax` (確率化)、`softmax_cross_entropy` (多値分類の損失) を実装。
2. ステップ 48 (多値分類) 正式な損失関数でスパイラルを分類。
3. ステップ 49 (Dataset) データを扱う統一的な入れ物。前処理 (正規化) も自動適用。
4. ステップ 50 (DataLoader) イテレータでミニバッチ供給を自動化。`accuracy` で評価。
5. ステップ 51 (MNIST) `ReLU` を実装し、全部品で手書き数字認識を実現。

### 第 4 ステージ全体の流れ
```
テンソル対応 (37-41)
   ↓
NN 構築フレームワーク：Layer/Model/Optimizer (42-46)
   ↓
【今回】実践的なデータ処理と本物のタスク (47-51)
   softmax/交差エントロピー → Dataset/DataLoader → MNIST 学習
```

### 完成した「実践的な学習ループ」
```python
train_loader = DataLoader(train_set, batch_size=100)
model = MLP((100, 10), activation=relu)
optimizer = MomentumSGD(lr=0.3).setup(model)

for epoch in range(max_epoch):
    for x, t in train_loader:              # ミニバッチ自動供給
        y = model(x)
        loss = softmax_cross_entropy(y, t)
        model.cleargrads()
        loss.backward()
        optimizer.update()
```
この形は、PyTorch の学習ループとほぼ完全に一致する。

### 次のステージ (第 5 ステージ) へ
第 4 ステージで、DeZero によるニューラルネットワーク構築の基礎が完成した。次の第 5 ステージ「DeZero で挑む」では、より高度なテーマへ進む。GPU 対応 (高速化)、モデルの保存と読み込み、Dropout (過学習対策)、そして画像認識で強力な CNN (畳み込みニューラルネットワーク) の実装へと発展していく。